# Depression-related Tweet Classification: N-gram Decision Trees

This is an exploratory text-label classification exercise, **not** a clinical depression diagnostic system.

In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# This notebook is a text-label classification exercise, not a clinical diagnostic tool.


In [ ]:
tweets_df = pd.read_csv('sentiment_tweets3.csv')
print(tweets_df[['message', 'label']].shape)
print(tweets_df['label'].value_counts().sort_index())
print('Missing tweet texts:', tweets_df['message'].isna().sum())

In [ ]:
# Reproduce the original 80/20 split with a fixed seed.
Xraw_train, Xraw_test, Yraw_train, Yraw_test = train_test_split(
    tweets_df['message'].fillna(''), tweets_df['label'], test_size=0.2, random_state=42)
# Report exact duplicate texts across partitions (potential train/test contamination).
train_texts = set(Xraw_train.astype(str).str.lower().str.strip())
test_texts = set(Xraw_test.astype(str).str.lower().str.strip())
print('Identical texts shared by train/test:', len(train_texts.intersection(test_texts)))

In [ ]:
def generate_N_grams(text, ngram=1):
    """Return adjacent n-grams after simple lowercase word tokenization and stopword removal."""
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower())
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return [' '.join(tokens[i:i + ngram]) for i in range(len(tokens) - ngram + 1)]


In [ ]:
def learn_vocabulary(X_train, y_train, n, top_k=5):
    """Learn ten features ONLY from the labeled training subset."""
    selected = []
    for label in (1, 0):
        counts = Counter()
        for message in X_train[y_train == label]:
            counts.update(generate_N_grams(message, n))
        # Deterministic ordering for ties; vocabulary never depends on test labels.
        top = sorted(counts, key=lambda phrase: (-counts[phrase], phrase))[:top_k]
        selected.extend(top)
    return selected


def transform_tweets(messages, vocabulary, n):
    """Apply the SAME training vocabulary to train and test text without using labels."""
    rows = []
    for message in messages:
        grams = set(generate_N_grams(message, n))
        rows.append([int(term in grams) for term in vocabulary])
    names = [f'dfeat{i}' for i in range(1, 6)] + [f'nfeat{i}' for i in range(1, 6)]
    return pd.DataFrame(rows, columns=names)


def make_features(X_train, X_test, y_train, n):
    vocabulary = learn_vocabulary(X_train, y_train, n)
    return (transform_tweets(X_train, vocabulary, n),
            transform_tweets(X_test, vocabulary, n),
            vocabulary)


In [ ]:
def Depression_predictor(X_train, X_test, y_train, y_test, label):
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    accuracy = accuracy_score(y_test, predictions)
    print(f"{label} accuracy: {accuracy:.6f}")
    print("Confusion matrix (rows=true, columns=predicted; labels 0, 1):")
    print(cm)
    print(classification_report(y_test, predictions, labels=[0, 1], zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1]).plot()
    plt.title(f"{label}: confusion matrix")
    plt.show()
    plt.figure(figsize=(16, 8))
    plot_tree(model, feature_names=X_train.columns, class_names=['0', '1'],
              filled=True, max_depth=3, fontsize=8)
    plt.title(f"{label}: decision tree (top four levels)")
    plt.show()
    return {"accuracy": accuracy, "TN": int(cm[0,0]), "FP": int(cm[0,1]),
            "FN": int(cm[1,0]), "TP": int(cm[1,1])}


In [ ]:
# The test labels are kept separate throughout feature selection and transformation.
feature_sets = {}
for n in (1, 2, 3):
    feature_sets[n] = make_features(Xraw_train, Xraw_test, Yraw_train, n)
    print(f"{n}-gram training vocabulary:", feature_sets[n][2])


## Unigrams (1-grams)

In [ ]:
results = {}
results[1] = Depression_predictor(feature_sets[1][0], feature_sets[1][1], Yraw_train, Yraw_test, 'Unigrams')

**Interpretation:** Feature selection uses only the training labels. The test set is transformed with the same fixed vocabulary without accessing its labels. All metrics below must be generated by running this corrected notebook; historical results from the previous method were invalid due to test-label leakage.

## Bigrams (2-grams)

In [ ]:
results[2] = Depression_predictor(feature_sets[2][0], feature_sets[2][1], Yraw_train, Yraw_test, 'Bigrams')

## Trigrams (3-grams)

In [ ]:
results[3] = Depression_predictor(feature_sets[3][0], feature_sets[3][1], Yraw_train, Yraw_test, 'Trigrams')

In [ ]:
summary = pd.DataFrame.from_dict(results, orient='index')
summary.index.name = 'n_grams'
print(summary.to_string())

## Dataset audit and label-word ablation

Most label-1 tweets explicitly contain the word **depression**, whereas the label-0 tweets do not. This makes the raw classification task nearly equivalent to detecting the label word, rather than recognizing general linguistic markers of mental health. We also test an ablation that masks this word in both classes. **Masking a single term does not remove all potential dataset construction biases.**

In [ ]:
# Dataset inspection: this is a dataset-level descriptive audit, not part of model fitting.
label_word = re.compile(r'\bdepression\b', flags=re.IGNORECASE)
audit = tweets_df.assign(
    contains_depression=tweets_df['message'].fillna('').astype(str).str.contains(
        label_word, regex=True))
print(pd.crosstab(audit['label'], audit['contains_depression']))
print('Unique text values:', tweets_df['message'].fillna('').astype(str).str.lower().str.strip().nunique())
print('Exact duplicate occurrences:', len(tweets_df) - tweets_df['message'].fillna('').astype(str).str.lower().str.strip().nunique())


In [ ]:
# Leakage-free ablation: mask the explicit label word on BOTH sides of the split.
def mask_label_word(series):
    return series.astype(str).str.replace(label_word, ' ', regex=True)

masked_train = mask_label_word(Xraw_train)
masked_test = mask_label_word(Xraw_test)
masked_results = {}
for n in (1, 2, 3):
    train_f, test_f, vocab = make_features(masked_train, masked_test, Yraw_train, n)
    masked_results[n] = Depression_predictor(train_f, test_f, Yraw_train, Yraw_test,
                                             f'Masked depression: {n}-grams')
print('Masked-label-word metrics:')
print(pd.DataFrame.from_dict(masked_results, orient='index').to_string())


In [ ]:
# Additional robustness check: group identical texts into the same partition.
# This split is NOT stratified; class proportions may differ slightly from the 80/20 experiment.
from sklearn.model_selection import GroupShuffleSplit
texts = tweets_df['message'].fillna('').astype(str)
labels = tweets_df['label']
groups = texts.str.lower().str.strip()
train_ix, test_ix = next(GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42).split(texts, labels, groups=groups))
group_train = mask_label_word(texts.iloc[train_ix])
group_test = mask_label_word(texts.iloc[test_ix])
group_y_train = labels.iloc[train_ix]
group_y_test = labels.iloc[test_ix]
assert not set(groups.iloc[train_ix]).intersection(set(groups.iloc[test_ix]))
group_train_f, group_test_f, group_vocab = make_features(
    group_train, group_test, group_y_train, 1)
print('Group-split training/test sizes:', len(train_ix), len(test_ix))
group_result = Depression_predictor(group_train_f, group_test_f, group_y_train, group_y_test,
                                   'Masked unigram, duplicate-grouped split')
print('Duplicate-grouped masked-unigram metrics:', group_result)
